# 🤖 AutoML Pilot - Training Template
Use this Colab notebook to run advanced AutoML (PyCaret) on your dataset. This template is designed to work seamlessly with datasets exported from the AutoML Pilot web app.

In [ ]:
# @title 1. Install Dependencies
# Install PyCaret and other necessary libraries
!pip install -q pycaret[full] ydata-profiling pandas
print('✅ Dependencies installed!')

In [ ]:
# @title 2. Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
print('✅ Drive mounted! You can now access your files at /content/drive/MyDrive/')

In [ ]:
# @title 3. Load Dataset
import pandas as pd
from pathlib import Path

# EDIT THIS PATH to point to your dataset
dataset_path = '/content/drive/MyDrive/dataset.csv' # @param {type:"string"}

if os.path.exists(dataset_path):
    df = pd.read_csv(dataset_path)
    print(f'✅ Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns')
    display(df.head())
else:
    print(f'❌ Error: File not found at {dataset_path}')

In [ ]:
# @title 4. Automated EDA (Optional)
from ydata_profiling import ProfileReport

run_eda = True # @param {type:"boolean"}
if run_eda:
    profile = ProfileReport(df, title="AutoML Pilot - Data Profile", explorative=True)
    profile.to_notebook_iframe()

In [ ]:
# @title 5. AutoML Setup & Training
from pycaret.classification import setup as cls_setup, compare_models as cls_compare, pull as cls_pull, save_model as cls_save, finalize_model as cls_finalize
from pycaret.regression import setup as reg_setup, compare_models as reg_compare, pull as reg_pull, save_model as reg_save, finalize_model as reg_finalize

# Configuration
target_column = 'target' # @param {type:"string"}
task_type = 'classification' # @param ["classification", "regression"]

print(f'🚀 Starting AutoML for {task_type} on target: {target_column}')

if task_type == 'classification':
    s = cls_setup(data=df, target=target_column, session_id=123, verbose=False, html=False)
    print('Comparing models...')
    best_model = cls_compare()
    leaderboard = cls_pull()
    display(leaderboard)
    
    # Finalize and Save
    final_model = cls_finalize(best_model)
    cls_save(final_model, 'best_classification_model')
    print('✅ Classification model saved as best_classification_model.pkl')

else:
    s = reg_setup(data=df, target=target_column, session_id=123, verbose=False, html=False)
    print('Comparing models...')
    best_model = reg_compare()
    leaderboard = reg_pull()
    display(leaderboard)
    
    # Finalize and Save
    final_model = reg_finalize(best_model)
    reg_save(final_model, 'best_regression_model')
    print('✅ Regression model saved as best_regression_model.pkl')

In [ ]:
# @title 6. Email Results (Optional)
import smtplib
import ssl
import json
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

send_email = False # @param {type:"boolean"}
recipient_email = "" # @param {type:"string"}
gmail_user = "" # @param {type:"string"}
gmail_password = "" # @param {type:"string"}

if send_email and recipient_email and gmail_user and gmail_password:
    try:
        msg = MIMEMultipart("alternative")
        msg["Subject"] = f"AutoML Pilot Colab Results - {task_type.capitalize()}"
        msg["From"] = f"AutoML Pilot <{gmail_user}>"
        msg["To"] = recipient_email

        results_html = leaderboard.head(5).to_html(classes='table table-striped')
        
        html_body = f"""
        <html>
        <body style='font-family:sans-serif; padding:20px;'>
          <div style='max-width:600px; margin:0 auto; background:white; padding:30px; border-radius:15px; border:1px solid #eee;'>
            <h1 style='color:#7c3aed;'>🚀 AutoML Pilot Pro</h1>
            <h3>Google Colab Training Report</h3>
            <p><strong>Task:</strong> {task_type}</p>
            <p><strong>Target:</strong> {target_column}</p>
            <h4>Top 5 Models:</h4>
            {results_html}
            <p style='margin-top:20px; color:#666;'>Report generated automatically from Google Colab.</p>
          </div>
        </body>
        </html>
        """
        msg.attach(MIMEText(html_body, "html"))
        
        context = ssl.create_default_context()
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=context) as server:
            server.login(gmail_user, gmail_password)
            server.sendmail(gmail_user, recipient_email, msg.as_string())
        print("✅ Results emailed successfully!")
    except Exception as e:
        print(f"❌ Failed to send email: {e}")
else:
    print("ℹ️ Email reporting skipped. Configure credentials to enable.")

In [ ]:
# @title 7. Download Model
from google.colab import files

model_name = 'best_classification_model.pkl' if task_type == 'classification' else 'best_regression_model.pkl'
if os.path.exists(model_name):
    files.download(model_name)
    print(f'✅ Triggered download for {model_name}')
else:
    # PyCaret might save without .pkl extension in some versions for the function call
    alt_name = model_name.replace('.pkl', '')
    if os.path.exists(alt_name + '.pkl'):
        files.download(alt_name + '.pkl')
        print(f'✅ Triggered download for {alt_name}.pkl')
    else:
        print('❌ Model file not found.')